In [15]:
!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage
#!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train

train  val


In [16]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

train_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
test_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"

BATCH_SIZE = 64
IMG_SIZE = (224,224)

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)


print("Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:\n")
print("class_names = [")
for name in train_ds.class_names:
    print(f'    "{name}",')
print("]")

Found 43444 files belonging to 38 classes.
Found 10861 files belonging to 38 classes.
Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:

class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squa

In [17]:
# from tensorflow.keras import layers, models

# AUTOTUNE = tf.data.AUTOTUNE
# train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
# test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# base_model = tf.keras.applications.MobileNetV2(
#     input_shape=(224,224,3),
#     include_top=False,
#     weights="imagenet"
# )

# base_model.trainable = False

# model = models.Sequential([
#     layers.Input(shape=(224, 224, 3)),

#     layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
#     layers.RandomFlip("horizontal"),
#     layers.RandomBrightness(0.05),
#     layers.RandomContrast(0.05),
    
#     base_model,
    
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation="relu"),
#     layers.Dropout(0.2),

#     layers.Dense(38, activation="softmax")
# ])

# model.compile(
#     optimizer="adam",
#     loss="sparse_categorical_crossentropy",
#     metrics=['accuracy']
# )

# model.summary()

import tensorflow as tf
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# 1. Definicja modelu bazowego
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Zamrażamy na pierwszą fazę

# 2. Definicja osobnego bloku augmentacji
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.05),
    layers.RandomContrast(0.05),
], name="data_augmentation")

# 3. Główna architektura sieci
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    # Preprocessing specyficzny dla MobileNetV2 (skalowanie pikseli)
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    
    # Warstwa augmentacji (Keras wyłączy ją automatycznie podczas testu/walidacji!)
    data_augmentation,
    
    # Model bazowy
    base_model,
    
    # Głowa klasyfikatora
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(38, activation="softmax")
])

# --- FAZA 1: Trening samej głowy klasyfikatora ---
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

model.summary()

print("--starting first stage--")
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=7
)

# # --- FAZA 2: Fine-tuning (Odmrażanie końcówki MobileNetV2) ---
# print("--preparing model for second stage--")

# # Zamiast szukać po indeksie warstwy, odwołujemy się bezpośrednio do zmiennej base_model
# base_model.trainable = True

# # Zamrażamy wszystkie warstwy base_model OPRÓCZ ostatnich 30
# for layer in base_model.layers[:-30]:
#     layer.trainable = False

# # Bardzo ważny krok: kompilacja z BARDZO MAŁYM learning rate
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

# print("--starting second stage - fine-tuning--")
# history_f = model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=5
# )

# # Zapisanie w pełni wyszkolonego modelu
# model_save_path = '/kaggle/working/plant_disease_detector_v3.keras'
# model.save(model_save_path)
# print(f"Model został pomyślnie zapisany w: {model_save_path}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 38)             │        19,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,933,350 (11.19 MB)

 Trainable params: 675,366 (2.58 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

--starting first stage--
Epoch 1/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 61s 76ms/step - accuracy: 0.2273 - loss: 2.9248 - val_accuracy: 0.7494 - val_loss: 0.8580
Epoch 2/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.3427 - loss: 2.3741 - val_accuracy: 0.8333 - val_loss: 0.5116
Epoch 3/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 40s 60ms/step - accuracy: 0.3759 - loss: 2.2595 - val_accuracy: 0.8297 - val_loss: 0.5272
Epoch 4/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.3937 - loss: 2.1958 - val_accuracy: 0.8189 - val_loss: 0.5284
Epoch 5/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 38s 56ms/step - accuracy: 0.3964 - loss: 2.1754 - val_accuracy: 0.8663 - val_loss: 0.3938
Epoch 6/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 37s 55ms/step - accuracy: 0.4022 - loss: 2.1598 - val_accuracy: 0.8539 - val_loss: 0.4261
Epoch 7/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 37s 55ms/step - accuracy: 0.4087 - loss: 2.1253 - val_accuracy: 0.8722 - val_loss: 0.3897


In [18]:
# print("--starting first stage--")

# history = model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=5
# )

In [19]:
# print("--preparing model for second stage--")

# unfreezeModel = model.layers[4]
# unfreezeModel.trainable = True

# for layer in unfreezeModel.layers[:-30]:
#     layer.trainable = False

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

In [20]:
# print("--starting second stage - fine-tuning--")

# history_f = model.fit(
#     train_ds,
#     validation_data = test_ds,
#     epochs=5
# )

In [21]:
# # Zapisanie wyszkolonego modelu 
# model_save_path = '/kaggle/working/plant_disease_detector_v3.keras'
# model.save(model_save_path)

# print(f"Model został pomyślnie zapisany w: {model_save_path}")

In [22]:
!pip install gradio
import gradio as gr
import numpy as np
import tensorflow as tf

import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

# Generujemy listę klas automatycznie z tej samej struktury katalogów
temp_ds = image_dataset_from_directory(
    "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val",
    image_size=(224, 224),
    batch_size=1
)
class_names = temp_ds.class_names


model_path = '/kaggle/working/plant_disease_detector_v3.keras'

model = tf.keras.models.load_model(
    model_path,
    custom_objects={
        'preprocess_input': tf.keras.applications.mobilenet_v2.preprocess_input
    }
)

# class_names = [
#     "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
#     "Blueberry___healthy", "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy",
#     "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
#     "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy", "Grape___Black_rot",
#     "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
#     "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
#     "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight",
#     "Potato___Late_blight", "Potato___healthy", "Raspberry___healthy", "Soybean___healthy",
#     "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
#     "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
#     "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
#     "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus",
#     "Tomato___healthy"
# ]


def predict_plant_disease(image):
    if image is None:
        return "Proszę przesłać zdjęcie."
    
    # Zmiana rozmiaru do formatu wejściowego sieci (224, 224)
    img_resized = tf.image.resize(image, (224, 224))
    
    # Dodanie wymiaru batcha: z (224, 224, 3) robi się (1, 224, 224, 3)
    # Nie robimy preprocess_input ręcznie, bo Twój model ma już warstwę Lambda!
    img_tensor = tf.expand_dims(img_resized, axis=0)
    
    # Wykonanie predykcji
    predictions = model.predict(img_tensor)[0]
    
    # Zwracamy słownik, gdzie kluczem jest nazwa klasy, a wartością prawdopodobieństwo
    # Gradio automatycznie ładnie to wyświetli w formie pasków procentowych
    return {class_names[i]: float(predictions[i]) for i in range(len(class_names))}



interface = gr.Interface(
    fn=predict_plant_disease,
    inputs=gr.Image(),
    #inputs=gr.Image(shape=(224, 224)), # Gradio wstępnie dotnie obraz, ale tf.image.resize w funkcji daje 100% pewności
    outputs=gr.Label(num_top_classes=3), # Pokaże 3 najbardziej prawdopodobne choroby
    title="Detektor Chorób Roślin (PlantVillage)",
    description="Wgraj zdjęcie liścia, aby sprawdzić stan zdrowia rośliny. Model oparty na MobileNetV2.",
    examples=[
        # Tutaj możesz podać ścieżki do przykładowych zdjęć z datasetu val, żeby mieć je pod ręką do kliknięcia
        # "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val/Potato___Early_blight/jakis_plik.jpg"
    ]
)

interface.launch(share=True)

Found 10861 files belonging to 38 classes.
* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://14f761ba75c5520b3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
# import tensorflow as tf
# from tensorflow.keras.utils import image_dataset_from_directory
# from tensorflow.keras import layers
# from tensorflow.keras.applications import MobileNetV2
# from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Input, BatchNormalization
# from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import Adam

# # === 1) Ładowanie i przygotowanie danych (Kategoryczne etykiety) ===
# train_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
# test_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"
# BATCH_SIZE = 64
# IMG_SIZE = (224, 224)
# num_classes = 38

# print("--- Ładowanie zbioru treningowego ---")
# train_ds = image_dataset_from_directory(
#     train_dir,
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     label_mode='categorical' # 🚀 ZMIANA: Wymagane dla categorical_crossentropy (One-Hot)
# )

# print("\n--- Ładowanie zbioru testowego ---")
# test_ds = image_dataset_from_directory(
#     test_dir,
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     label_mode='categorical' # 🚀 ZMIANA: Wymagane dla categorical_crossentropy (One-Hot)
# )

# AUTOTUNE = tf.data.AUTOTUNE
# train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
# test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


# # === 2) Budowa architektury Funkcjonalnej (Wersja Prawidłowa) ===
# print("\n--- Inicjalizacja architektury funkcjonalnej ---")

# BASE_LR = 1e-3

# # 1. Definiujemy czyste wejście dla całego modelu
# inputs = Input(shape=(224, 224, 3))

# # 2. Tworzymy sekwencyjny łańcuch przetwarzania obrazu
# x = layers.RandomFlip("horizontal")(inputs)
# x = layers.RandomBrightness(0.05)(x)
# x = layers.RandomContrast(0.05)(x)
# x = layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input)(x)

# # 3. Ładujemy czysty, odizolowany szkielet MobileNetV2 z ImageNet
# base_model = MobileNetV2(
#     include_top=False,
#     weights="imagenet",
#     input_shape=(224, 224, 3) # Definiujemy standardowy kształt wejściowy
# )
# base_model.trainable = False # Zamrażamy bazę dla pierwszej fazy

# # 4. KLUCZOWA POPRAWKA: Przepuszczamy nasz przetworzony tensor 'x' 
# # PRZEZ base_model jak przez zwykłą warstwę. To gwarantuje, że Keras 
# # nigdy nie odetnie preprocessingu w trybie testowym Gradio!
# x_backbone = base_model(x)

# # 5. Budowa nowej głowy klasyfikatora
# x_head = GlobalAveragePooling2D()(x_backbone)
# x_head = Dropout(0.3)(x_head)
# outputs = Dense(num_classes, activation="softmax")(x_head)

# # Spięcie całości w jeden funkcjonalny Model
# model = Model(inputs=inputs, outputs=outputs)

# model.compile(
#     optimizer=Adam(learning_rate=BASE_LR),
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )

# model.summary()


# # === 3) Trening: Faza I (Zamrożony Backbone) ===
# print("\n-- starting first stage --")
# history = model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=5
# )


# # === 4) Trening: Faza II (Fine-Tuning - Odmrażanie warstw końcowych) ===
# print("\n-- preparing model for second stage - fine-tuning --")

# # Odmrażamy całą strukturę MobileNetV2
# base_model.trainable = True

# # 🚀 ZMIANA: Bezpieczna pętla fine-tuningu. Zamrażamy początek, odmrażamy koniec,
# # ale BEZWZGLĘDNIE blokujemy modyfikację warstw BatchNormalization (BN),
# # ponieważ rozregulowałyby wyuczone filtry ImageNet.
# for layer in base_model.layers[:-30]:
#     layer.trainable = False

# for layer in base_model.layers[-30:]:
#     if isinstance(layer, BatchNormalization):
#         layer.trainable = False # BN zostaje zamrożone dla stabilności wag
#     else:
#         layer.trainable = True

# # Kompilacja z niższym learning rate dla delikatnego dostrajania
# FINE_TUNE_LR = 1e-4
# model.compile(
#     optimizer=Adam(learning_rate=FINE_TUNE_LR),
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )

# model.summary()

# print("\n-- starting second stage - fine-tuning --")
# history_f = model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=7
# )


# # === 5) Zapisanie modelu ===
# model_save_path = '/kaggle/working/plant_disease_detector_v5.keras'
# model.save(model_save_path)
# print(f"\n✅ Nowy, funkcjonalny model został pomyślnie zapisany w: {model_save_path}")

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Created dataset file at: .gradio/flagged/dataset1.csv


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error